In [ ]:
# this code is based on Andrej Karpathy's "Let's build GPT from scratch" Youtube lecture
  # (another dataset was used here - see alllines_processed.txt)
# https://www.youtube.com/watch?v=kCc8FmEb1nY&list=PLAqhIrjkxbuWI23v9cThsA9GvCAUhRvKZ&index=7

In [75]:
!wget https://raw.githubusercontent.com/ruxandrailiescu/mini-gpt/main/data/alllines_processed.txt

--2026-01-11 23:30:13--  https://raw.githubusercontent.com/ruxandrailiescu/mini-gpt/main/alllines_processed.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.111.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 4871678 (4.6M) [text/plain]
Saving to: ‘alllines_processed.txt.3’

alllines_processed. 100%[===================>]   4.65M  --.-KB/s    in 0.05s   

2026-01-11 23:30:14 (86.6 MB/s) - ‘alllines_processed.txt.3’ saved [4871678/4871678]



In [76]:
# the data was preprocessed using the method preprocess_data from preprocess.py
# the idea was to keep the character's name when they begin their lines so that the model
# could remember how a character "behaves" (Romeo is emotional, etc.)
with open('alllines_processed.txt', 'r') as f:
  text = f.read()

print(text[:2000])


KING HENRY IV: So shaken as we are, so wan with care,
		Find we a time for frighted peace to pant,
		And breathe short-winded accents of new broils
		To be commenced in strands afar remote.
		No more the thirsty entrance of this soil
		Shall daub her lips with her own children's blood,
		Nor more shall trenching war channel her fields,
		Nor bruise her flowerets with the armed hoofs
		Of hostile paces: those opposed eyes,
		Which, like the meteors of a troubled heaven,
		All of one nature, of one substance bred,
		Did lately meet in the intestine shock
		And furious close of civil butchery
		Shall now, in mutual well-beseeming ranks,
		March all one way and be no more opposed
		Against acquaintance, kindred and allies:
		The edge of war, like an ill-sheathed knife,
		No more shall cut his master. Therefore, friends,
		As far as to the sepulchre of Christ,
		Whose soldier now, under whose blessed cross
		We are impressed and engaged to fight,
		Forthwith a power of English shall we lev

In [92]:
import torch
import torch.nn as nn
from torch.nn import functional as F
import math

torch.manual_seed(5454)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
batch_size = 32
context_length = 256
max_iters = 10000
eval_interval = 500
learning_rate = 3e-4
eval_iters = 200
n_embed = 384
n_head = 6
n_layer = 6
dropout = 0.2

In [93]:
# character tokenizer
chars = sorted(list(set(text)))
print(len(chars))
print(''.join(chars))   # also have '\n', '\t'
stoi = {ch:i for i,ch in enumerate(chars)}
itos = {i:ch for i,ch in enumerate(chars)}
print(stoi)
print(itos)

vocab_size = len(chars)

77
	
 !$'(),-.0123456789:?ABCDEFGHIJKLMNOPQRSTUVWXYZ[]abcdefghijklmnopqrstuvwxyz
{'\t': 0, '\n': 1, ' ': 2, '!': 3, '$': 4, "'": 5, '(': 6, ')': 7, ',': 8, '-': 9, '.': 10, '0': 11, '1': 12, '2': 13, '3': 14, '4': 15, '5': 16, '6': 17, '7': 18, '8': 19, '9': 20, ':': 21, '?': 22, 'A': 23, 'B': 24, 'C': 25, 'D': 26, 'E': 27, 'F': 28, 'G': 29, 'H': 30, 'I': 31, 'J': 32, 'K': 33, 'L': 34, 'M': 35, 'N': 36, 'O': 37, 'P': 38, 'Q': 39, 'R': 40, 'S': 41, 'T': 42, 'U': 43, 'V': 44, 'W': 45, 'X': 46, 'Y': 47, 'Z': 48, '[': 49, ']': 50, 'a': 51, 'b': 52, 'c': 53, 'd': 54, 'e': 55, 'f': 56, 'g': 57, 'h': 58, 'i': 59, 'j': 60, 'k': 61, 'l': 62, 'm': 63, 'n': 64, 'o': 65, 'p': 66, 'q': 67, 'r': 68, 's': 69, 't': 70, 'u': 71, 'v': 72, 'w': 73, 'x': 74, 'y': 75, 'z': 76}
{0: '\t', 1: '\n', 2: ' ', 3: '!', 4: '$', 5: "'", 6: '(', 7: ')', 8: ',', 9: '-', 10: '.', 11: '0', 12: '1', 13: '2', 14: '3', 15: '4', 16: '5', 17: '6', 18: '7', 19: '8', 20: '9', 21: ':', 22: '?', 23: 'A', 24: 'B', 25: 'C', 26: 'D

In [94]:
# encode-decode
encode = lambda s: [stoi[ch] for ch in s]
decode = lambda ints: ''.join([itos[i] for i in ints])

test_str = 'hello world'
print(encode(test_str))
print(decode(encode(test_str)))

[58, 55, 62, 62, 65, 2, 73, 65, 68, 62, 54]
hello world


In [95]:
# train-test split (90-10)
data = torch.tensor(encode(text), dtype=torch.long)
print(data[:100])
print(data.shape)

n = int(0.9*len(data))
train = data[:n+1]
test = data[n+1:]

print(train.shape)
print(test.shape)
print(decode(train[-10:].tolist()))   # check for data leakage
print(decode(test[:10].tolist()))

tensor([ 1, 33, 31, 36, 29,  2, 30, 27, 36, 40, 47,  2, 31, 44, 21,  2, 41, 65,
         2, 69, 58, 51, 61, 55, 64,  2, 51, 69,  2, 73, 55,  2, 51, 68, 55,  8,
         2, 69, 65,  2, 73, 51, 64,  2, 73, 59, 70, 58,  2, 53, 51, 68, 55,  8,
         1,  0,  0, 28, 59, 64, 54,  2, 73, 55,  2, 51,  2, 70, 59, 63, 55,  2,
        56, 65, 68,  2, 56, 68, 59, 57, 58, 70, 55, 54,  2, 66, 55, 51, 53, 55,
         2, 70, 65,  2, 66, 51, 64, 70,  8,  1])
torch.Size([4871678])
torch.Size([4384511])
torch.Size([487167])
 learn me 
the procla


In [96]:
# load a batch of data
def get_batch(split):
  data = train if split == 'train' else test
  ix = torch.randint(len(data) - context_length, (batch_size,))
  x = torch.stack([data[i:i+context_length] for i in ix])
  y = torch.stack([data[i+1:i+context_length+1] for i in ix])
  x, y = x.to(device), y.to(device)
  return x, y

In [97]:
# implementation of nn modules
class LinearLayer(nn.Module):
  """ affine linear transformation """

  def __init__(self, in_features, out_features, bias=True):
    super().__init__()
    self.in_features = in_features
    self.out_features = out_features
    self.weight = nn.Parameter(torch.randn(out_features, in_features))

    if bias:
      self.bias = nn.Parameter(torch.zeros(out_features))
    else:
      self.register_parameter('bias', None)

    self.reset_parameters()

  def reset_parameters(self):
    sqrt_k = 1. / math.sqrt(self.in_features)
    self.weight.data.uniform_(-sqrt_k, sqrt_k)
    if self.bias is not None:
      self.bias.data.uniform_(-sqrt_k, sqrt_k)

  def forward(self, x):
    x = x @ self.weight.T
    if self.bias is not None:
      x = x + self.bias
    return x


class Head(nn.Module):
  """ single head of self-attention """

  def __init__(self, head_size):
    super().__init__()
    self.key = LinearLayer(n_embed, head_size, bias=False)  # using the custom linear layer
    self.query = LinearLayer(n_embed, head_size, bias=False)
    self.value = LinearLayer(n_embed, head_size, bias=False)
    self.register_buffer('tril', torch.tril(torch.ones(context_length, context_length)))  # attention mask
    self.dropout = nn.Dropout(dropout)

  def forward(self, x):
    # input: (batch, time-step, embed)
    # output: (batch, time-step, head_size)
    b,t,e = x.shape
    k = self.key(x) # (b, t, hs)
    q = self.query(x) # (b, t, hs)

    # scaled dot-attention
    w = q @ k.transpose(-2,-1) * k.shape[-1]**-0.5 # (b, t, hs) @ (b, t, hs) --> (b, t, t)
    w = w.masked_fill(self.tril[:t, :t] == 0, float('-inf'))
    w = F.softmax(w, dim=-1)
    w = self.dropout(w)

    v = self.value(x) # (b, t, hs)
    out = w @ v # (b, t, t) @ (b, t, hs) --> (b, t, hs)
    return out


class MultiHeadAttention(nn.Module):
  """ multiple self-attention heads in parallel """

  def __init__(self, num_heads, head_size):
    super().__init__()
    self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
    self.proj = LinearLayer(head_size * num_heads, n_embed)
    self.dropout = nn.Dropout(dropout)

  def forward(self, x):
    out = torch.cat([h(x) for h in self.heads], dim=-1)
    out = self.dropout(self.proj(out))
    return out


class FeedForward(nn.Module):
  """ feed-forward network with 2 layers and a non-linear activation in between """

  def __init__(self, n_embed):
    super().__init__()
    self.net = nn.Sequential(
        LinearLayer(n_embed, n_embed*4),
        nn.ReLU(),
        LinearLayer(n_embed*4, n_embed),
        nn.Dropout(dropout),
    )

  def forward(self, x):
    return self.net(x)


class LayerNormalization(nn.Module):
  """ computes statistics across a single sample (features dimension) """

  def __init__(self, dim, eps=1e-5):
    super().__init__()
    self.eps = eps
    self.gamma = nn.Parameter(torch.ones(dim))
    self.beta = nn.Parameter(torch.zeros(dim))

  def forward(self, x):
    xmean = x.mean(-1, keepdim=True)
    xvar = x.var(-1, keepdim=True)
    xhat = (x - xmean) / torch.sqrt(xvar + self.eps)
    self.out = self.gamma * xhat + self.beta
    return self.out


class Block(nn.Module):
  """ transformer block """

  def __init__(self, n_embed, n_head):
    super().__init__()
    head_size = n_embed // n_head
    self.mha = MultiHeadAttention(n_head, head_size)
    self.ffn = FeedForward(n_embed)
    self.ln1 = LayerNormalization(n_embed)
    self.ln2 = LayerNormalization(n_embed)

  def forward(self, x):
    x = x + self.mha(self.ln1(x))
    x = x + self.ffn(self.ln2(x))
    return x


class MiniGPT(nn.Module):
  """ entire decoder applying the transformer block times n_layer"""

  def __init__(self):
    super().__init__()
    self.token_embed_table = nn.Embedding(vocab_size, n_embed)
    self.position_embed_table = nn.Embedding(context_length, n_embed)
    self.blocks = nn.Sequential(*[Block(n_embed, n_head) for _ in range(n_layer)])
    self.lin = LinearLayer(n_embed, vocab_size)

  def forward(self, idx, targets=None):
    b, t = idx.shape
    tok_emb = self.token_embed_table(idx) # (b, t, e)
    pos_emb = self.position_embed_table(torch.arange(t, device=device)) # (t, e)
    x = tok_emb + pos_emb # (b, t, e)
    x = self.blocks(x) # (b, t, e)
    logits = self.lin(x) # (b, t, vocab_size)

    if targets is None:
        loss = None
    else:
        b, t, vs = logits.shape
        logits = logits.view(b*t, vs)
        targets = targets.view(b*t)
        loss = F.cross_entropy(logits, targets)

    return logits, loss

  def generate(self, idx, max_new_tokens, temperature=1.0):
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -context_length:]
        logits, loss = self(idx_cond)
        logits = logits[:, -1, :] / temperature # (b, vs)
        probs = F.softmax(logits, dim=-1) # (b, vs)
        idx_next = torch.multinomial(probs, num_samples=1) # (b, 1)
        idx = torch.cat((idx, idx_next), dim=1) # (b, t+1)
    return idx

In [98]:
# check forward method of linear layer
m1 = LinearLayer(12, 32)
m2 = nn.Linear(12, 32)
m1.weight.data = m2.weight.data
m1.bias.data = m2.bias.data
x = torch.randn(4, 8, 12)
out1 = m1(x)
out2 = m2(x)

if torch.allclose(out1, out2):
  print("custom forward method works like pytorch")

custom forward method works like pytorch


In [99]:
model = MiniGPT()
m = model.to(device)
print(sum(p.numel() for p in m.parameters())/1e6, 'M parameters')
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

10.797389 M parameters


In [100]:
# calculate loss and perplexity
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        mean_loss = losses.mean().item()
        perplexity = math.exp(mean_loss)
        out[split] = {
            'loss': mean_loss,
            'perplexity': perplexity
        }
    model.train()
    return out

In [101]:
results = []
for iter in range(max_iters):
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        results.append(losses)
        print(f"step {iter}: train loss {losses['train']['loss']:.4f}, train perplexity {losses['train']['perplexity']:.4f} \
         val loss {losses['val']['loss']:.4f}, val perplexity {losses['val']['perplexity']:.4f}")

    xb, yb = get_batch('train')

    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

step 0: train loss 4.7017, train perplexity 110.1312          val loss 4.6974, val perplexity 109.6592
step 500: train loss 2.0464, train perplexity 7.7402          val loss 2.0637, val perplexity 7.8747
step 1000: train loss 1.7214, train perplexity 5.5925          val loss 1.7688, val perplexity 5.8638
step 1500: train loss 1.5630, train perplexity 4.7733          val loss 1.6486, val perplexity 5.1995
step 2000: train loss 1.4708, train perplexity 4.3525          val loss 1.5810, val perplexity 4.8600
step 2500: train loss 1.4063, train perplexity 4.0807          val loss 1.5337, val perplexity 4.6354
step 3000: train loss 1.3578, train perplexity 3.8878          val loss 1.5033, val perplexity 4.4965
step 3500: train loss 1.3219, train perplexity 3.7506          val loss 1.4755, val perplexity 4.3734
step 4000: train loss 1.2931, train perplexity 3.6440          val loss 1.4693, val perplexity 4.3461
step 4500: train loss 1.2678, train perplexity 3.5529          val loss 1.4424, va

In [102]:
# generate from the model
context = torch.zeros((1, 1), dtype=torch.long, device=device)
hamlet_sample1 = "HAMLET: Not so, my lord, I am too much i' the sun.\n\nROMEO: "
hamlet_sample2 = "HAMLET: To be, or not to be, that is the question.\n\nROMEO: "
context = torch.tensor(encode(hamlet_sample2), dtype=torch.long).unsqueeze(0).to(device)
# print(decode(m.generate(context, max_new_tokens=1000)[0].tolist()))
open('char_large_sample.txt', 'w').write(decode(m.generate(context, max_new_tokens=10000)[0].tolist()))

10059

In [103]:
# save loss dict
import json

with open('char_large_results.json', 'w') as f:
    json.dump(results, f, indent=4)